In [3]:
## SONGS WAV and AIFF
# -----######-----###### FILE SYSTEM TYPE BREAKDOWN IN GB -----######-----###### #
import os
import pandas as pd
from collections import defaultdict

def _fs_1807_breakdown_GET_types_gbtotal(folder_path):
    """
    Walks a folder recursively and returns a breakdown of:
    - File count and total size per extension (in GB)
    - Grand total size in GB
    """
    ext_sizes = defaultdict(float)
    ext_counts = defaultdict(int)

    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.startswith('._') or f in ['.DS_Store']:
                continue
            ext = os.path.splitext(f)[1].lower().strip('.')
            f_path = os.path.join(root, f)
            try:
                size_bytes = os.path.getsize(f_path)
                size_gb = size_bytes / (1024**3)
                ext_sizes[ext] += size_gb
                ext_counts[ext] += 1
            except Exception as e:
                print(f"⚠️ Skipping: {f_path} | {e}")

    df_stats = pd.DataFrame({
        'Extension': list(ext_sizes.keys()),
        'File Count': list(ext_counts.values()),
        'Size (GB)': [round(size, 4) for size in ext_sizes.values()]
    }).sort_values('Size (GB)', ascending=False).reset_index(drop=True)

    grand_total_gb = round(sum(ext_sizes.values()), 4)
    print(f"\n🎯 Grand Total Size: {grand_total_gb} GB")

    return df_stats
folder_path = "/Volumes/MUSIC_PROD/_5_MISC-zarch/_SINC_music/Contents"
df_breakdown = _fs_1807_breakdown_GET_types_gbtotal(folder_path)
print(df_breakdown)



🎯 Grand Total Size: 198.5529 GB
  Extension  File Count  Size (GB)
0       mp3       17327   131.5420
1      flac        1166    49.2038
2      aiff         137    10.3902
3       wav          94     4.8889
4       aif          33     2.3230
5       m4a          19     0.2050


In [5]:
# -----######-----###### GET PATHS & SIZES FOR SPECIFIC EXTENSIONS -----######-----###### #
import os
import pandas as pd

def _fs_1807_extpaths_GET_df_pathsizes(folder_path, target_extensions):
    """
    Finds all files with extensions in `target_extensions` inside `folder_path`.
    Returns a DataFrame with columns: Path, Extension, Size (GB)
    """
    paths, exts, sizes = [], [], []

    target_extensions = [ext.lower() for ext in target_extensions]

    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.startswith('._') or f == '.DS_Store':
                continue
            ext = os.path.splitext(f)[1].lower().strip('.')
            if ext in target_extensions:
                f_path = os.path.join(root, f)
                try:
                    size_gb = os.path.getsize(f_path) / (1024**3)
                    paths.append(f_path)
                    exts.append(ext)
                    sizes.append(round(size_gb, 5))
                except Exception as e:
                    print(f"⚠️ Skipping: {f_path} | {e}")

    df = pd.DataFrame({'Path': paths, 'Extension': exts, 'Size (GB)': sizes})
    df = df.sort_values('Size (GB)', ascending=False).reset_index(drop=True)
    return df
folder_path = "/Volumes/MUSIC_PROD/_5_MISC-zarch/_SINC_music/Contents"
target_extensions = ['flac', 'aiff', 'wav', 'aif']
df = _fs_1807_extpaths_GET_df_pathsizes(folder_path, target_extensions)
df

,Path,Extension,Size (GB)
0,/Volumes/MUSIC_PROD/_5_MISC-zarch/_SINC_music/...,wav,0.22369
1,/Volumes/MUSIC_PROD/_5_MISC-zarch/_SINC_music/...,wav,0.18603
2,/Volumes/MUSIC_PROD/_5_MISC-zarch/_SINC_music/...,wav,0.16556
3,/Volumes/MUSIC_PROD/_5_MISC-zarch/_SINC_music/...,aif,0.14404
4,/Volumes/MUSIC_PROD/_5_MISC-zarch/_SINC_music/...,aiff,0.14383
...,...,...,...
1425,/Volumes/MUSIC_PROD/_5_MISC-zarch/_SINC_music/...,flac,0.00099
1426,/Volumes/MUSIC_PROD/_5_MISC-zarch/_SINC_music/...,flac,0.00097
1427,/Volumes/MUSIC_PROD/_5_MISC-zarch/_SINC_music/...,flac,0.00097
1428,/Volumes/MUSIC_PROD/_5_MISC-zarch/_SINC_music/...,flac,0.00082


In [ ]:
####

In [10]:
# -----######-----###### EXTENSION-BASED CHUNKED COPY WITH LOG -----######-----###### #
import os
import shutil
import pandas as pd
from tqdm import tqdm

def _fileops_1807_chunkcopy_GET_extchunk_sorted(df, dest_base_path):
    """
    Groups by Extension, copies files in chunks of 50 to folders like _24_ARCH_rk_<ext>_<chunk#>.
    Ensures no filename collisions. Adds 'Copied_To' column with TQM progress bar.
    """
    tqdm.pandas()
    os.makedirs(dest_base_path, exist_ok=True)
    df = df.copy()
    
    # Ensure required columns
    df['file_name'] = df['Path'].apply(os.path.basename)
    df['Extension'] = df['Extension'].str.lower()
    
    copied_to = [None] * len(df)
    tq_bar = tqdm(total=len(df), desc="🎯 COPY FLOW", ncols=100)

    folder_index_by_ext = {}

    for ext, group in df.groupby('Extension'):
        group = group.sort_values(by='Size (GB)', ascending=False).reset_index()
        chunk_files = set()
        current_chunk = []
        chunk_counter = 1
        
        for _, row in group.iterrows():
            src = row['Path']
            fname = row['file_name']
            orig_idx = row['index']

            # Start new chunk if name conflict or chunk full
            if fname in chunk_files or len(current_chunk) >= 50:
                folder_name = f"_25_SINC_rk_{ext}_{chunk_counter}"
                dest_folder = os.path.join(dest_base_path, folder_name)
                os.makedirs(dest_folder, exist_ok=True)

                for p, n, i in current_chunk:
                    dest_path = os.path.join(dest_folder, n)
                    try:
                        shutil.copy2(p, dest_path)
                        copied_to[i] = dest_path
                        tq_bar.update(1)
                    except Exception as e:
                        tqdm.write(f"❌ Error copying {p} → {dest_path} | {e}")

                tqdm.write(f"📦 Finished: {folder_name} ({len(current_chunk)} files)")
                chunk_counter += 1
                chunk_files = set()
                current_chunk = []

            chunk_files.add(fname)
            current_chunk.append((src, fname, orig_idx))

        # Final chunk
        if current_chunk:
            folder_name = f"_25_SINC_rk_{ext}_{chunk_counter}"
            dest_folder = os.path.join(dest_base_path, folder_name)
            os.makedirs(dest_folder, exist_ok=True)

            for p, n, i in current_chunk:
                dest_path = os.path.join(dest_folder, n)
                try:
                    shutil.copy2(p, dest_path)
                    copied_to[i] = dest_path
                    tq_bar.update(1)
                except Exception as e:
                    tqdm.write(f"❌ Error copying {p} → {dest_path} | {e}")

            tqdm.write(f"📦 Final: {folder_name} ({len(current_chunk)} files)")

    tq_bar.close()
    df['Copied_To'] = copied_to

    # Save log
    log_path = os.path.join(dest_base_path, "copied_log.csv")
    df.to_csv(log_path, index=False)
    print(f"\n✅ Done. Log saved at: {log_path}")
    return df


In [11]:
dest_base_path = "/Volumes/MUSIC_PROD/_5_MISC-zarch/_music_SIN"
df = _fileops_1807_chunkcopy_GET_extchunk_sorted(df, dest_base_path)




🎯 COPY FLOW:   0%|                                                        | 0/1430 [00:00<?, ?it/s]

🎯 COPY FLOW:   0%|                                              | 1/1430 [00:11<4:39:06, 11.72s/it]

🎯 COPY FLOW:   0%|                                              | 2/1430 [00:17<3:21:36,  8.47s/it]

🎯 COPY FLOW:   0%|                                              | 3/1430 [00:23<2:52:24,  7.25s/it]

🎯 COPY FLOW:   0%|▏                                             | 4/1430 [00:29<2:36:41,  6.59s/it]

🎯 COPY FLOW:   0%|▏                                             | 5/1430 [00:38<2:55:01,  7.37s/it]

🎯 COPY FLOW:   0%|▏                                             | 6/1430 [00:43<2:37:16,  6.63s/it]

🎯 COPY FLOW:   0%|▏                                             | 7/1430 [00:48<2:25:30,  6.14s/it]

🎯 COPY FLOW:   1%|▎                                             | 8/1430 [00:53<2:16:46,  5.77s/it]

🎯 COPY FLOW:   1%|▎                                             | 9/1430 [00:58<2:09:18, 

📦 Final: _25_SINC_rk_aif_1 (33 files)




🎯 COPY FLOW:   2%|█                                            | 34/1430 [02:48<1:58:43,  5.10s/it]

🎯 COPY FLOW:   2%|█                                            | 35/1430 [02:56<2:21:35,  6.09s/it]

🎯 COPY FLOW:   3%|█▏                                           | 36/1430 [03:07<2:55:14,  7.54s/it]

🎯 COPY FLOW:   3%|█▏                                           | 37/1430 [03:15<2:58:27,  7.69s/it]

🎯 COPY FLOW:   3%|█▏                                           | 38/1430 [03:22<2:56:48,  7.62s/it]

🎯 COPY FLOW:   3%|█▏                                           | 39/1430 [03:30<2:52:46,  7.45s/it]

🎯 COPY FLOW:   3%|█▎                                           | 40/1430 [03:39<3:09:08,  8.16s/it]

🎯 COPY FLOW:   3%|█▎                                           | 41/1430 [03:46<2:58:14,  7.70s/it]

🎯 COPY FLOW:   3%|█▎                                           | 42/1430 [03:53<2:50:15,  7.36s/it]

🎯 COPY FLOW:   3%|█▎                                           | 43/1430 [03:59<2:43:34, 

📦 Finished: _25_SINC_rk_aiff_1 (50 files)




🎯 COPY FLOW:   6%|██▋                                          | 84/1430 [07:47<1:55:41,  5.16s/it]

🎯 COPY FLOW:   6%|██▋                                          | 85/1430 [07:52<1:52:06,  5.00s/it]

🎯 COPY FLOW:   6%|██▋                                          | 86/1430 [07:56<1:49:31,  4.89s/it]

🎯 COPY FLOW:   6%|██▋                                          | 87/1430 [08:01<1:47:38,  4.81s/it]

🎯 COPY FLOW:   6%|██▊                                          | 88/1430 [08:09<2:06:47,  5.67s/it]

🎯 COPY FLOW:   6%|██▊                                          | 89/1430 [08:13<1:59:39,  5.35s/it]

🎯 COPY FLOW:   6%|██▊                                          | 90/1430 [08:18<1:54:36,  5.13s/it]

🎯 COPY FLOW:   6%|██▊                                          | 91/1430 [08:23<1:51:04,  4.98s/it]

🎯 COPY FLOW:   6%|██▉                                          | 92/1430 [08:27<1:48:12,  4.85s/it]

🎯 COPY FLOW:   7%|██▉                                          | 93/1430 [08:32<1:45:56, 

...

📦 Final: _25_SINC_rk_flac_24 (16 files)




🎯 COPY FLOW:  93%|████████████████████████████████████████▏  | 1337/1430 [1:08:26<00:05, 15.54it/s]

🎯 COPY FLOW:  94%|████████████████████████████████████████▏  | 1338/1430 [1:08:39<06:42,  4.37s/it]

🎯 COPY FLOW:  94%|████████████████████████████████████████▎  | 1339/1430 [1:08:49<08:12,  5.41s/it]

🎯 COPY FLOW:  94%|████████████████████████████████████████▎  | 1340/1430 [1:08:58<08:58,  5.98s/it]

🎯 COPY FLOW:  94%|████████████████████████████████████████▎  | 1342/1430 [1:09:15<10:10,  6.94s/it]

🎯 COPY FLOW:  94%|████████████████████████████████████████▍  | 1343/1430 [1:09:21<09:44,  6.72s/it]

🎯 COPY FLOW:  94%|████████████████████████████████████████▍  | 1344/1430 [1:09:27<09:24,  6.56s/it]

🎯 COPY FLOW:  94%|████████████████████████████████████████▍  | 1345/1430 [1:09:35<09:56,  7.01s/it]

🎯 COPY FLOW:  94%|████████████████████████████████████████▍  | 1346/1430 [1:09:41<09:18,  6.65s/it]

🎯 COPY FLOW:  94%|████████████████████████████████████████▌  | 1347/1430 [1:09:46<08:43, 

📦 Finished: _25_SINC_rk_wav_1 (50 files)




🎯 COPY FLOW:  97%|█████████████████████████████████████████▋ | 1387/1430 [1:12:14<01:49,  2.55s/it]

🎯 COPY FLOW:  97%|█████████████████████████████████████████▋ | 1388/1430 [1:12:17<01:44,  2.49s/it]

🎯 COPY FLOW:  97%|█████████████████████████████████████████▊ | 1389/1430 [1:12:19<01:42,  2.50s/it]

🎯 COPY FLOW:  97%|█████████████████████████████████████████▊ | 1390/1430 [1:12:22<01:38,  2.46s/it]

🎯 COPY FLOW:  97%|█████████████████████████████████████████▊ | 1391/1430 [1:12:24<01:34,  2.42s/it]

🎯 COPY FLOW:  97%|█████████████████████████████████████████▊ | 1392/1430 [1:12:26<01:30,  2.38s/it]

🎯 COPY FLOW:  97%|█████████████████████████████████████████▉ | 1393/1430 [1:12:29<01:26,  2.33s/it]

🎯 COPY FLOW:  97%|█████████████████████████████████████████▉ | 1394/1430 [1:12:31<01:22,  2.29s/it]

🎯 COPY FLOW:  98%|█████████████████████████████████████████▉ | 1395/1430 [1:12:36<01:46,  3.05s/it]

🎯 COPY FLOW:  98%|█████████████████████████████████████████▉ | 1396/1430 [1:12:38<01:35, 

📦 Final: _25_SINC_rk_wav_2 (44 files)

✅ Done. Log saved at: /Volumes/MUSIC_PROD/_5_MISC-zarch/_music_SIN/copied_log.csv


In [18]:
# -----######-----###### UNIVERSAL TAB-DELIM IMPORT FIXER -----######-----######
import pandas as pd
import chardet
from collections import Counter

def _readfix_2304_txt_GET_df_autoclean(path_txt):
    """
    Reads a tab-delimited text file with inconsistent columns.
    Automatically:
    - Detects encoding
    - Finds dominant column count
    - Truncates or pads rows
    - Returns clean DataFrame
    """
    # Detect encoding
    with open(path_txt, 'rb') as f:
        raw_data = f.read()
        encoding = chardet.detect(raw_data)['encoding']

    lines = raw_data.decode(encoding).splitlines()

    # Split lines and count how many fields in each
    split_lines = [line.strip().split('\t') for line in lines]
    col_lengths = [len(row) for row in split_lines]
    count_freq = Counter(col_lengths)

    # Determine most common column count
    most_common_len = count_freq.most_common(1)[0][0]

    print(f"\n🧠 Most common column count: {most_common_len}")
    print(f"📊 Full column count distribution:")
    for length, freq in sorted(count_freq.items()):
        print(f"{length} columns → {freq} lines")

    # Use first row (or generate header) and fix all rows to match length
    header = split_lines[0]
    if len(header) != most_common_len:
        print("⚠️ Header length mismatch — fixing with auto header")
        header = [f'col_{i}' for i in range(most_common_len)]
        start_idx = 0
    else:
        start_idx = 1

    fixed_rows = []
    for row in split_lines[start_idx:]:
        if len(row) < most_common_len:
            row.extend([''] * (most_common_len - len(row)))
        elif len(row) > most_common_len:
            row = row[:most_common_len]
        fixed_rows.append(row)

    return pd.DataFrame(fixed_rows, columns=header)

# -----######-----######-----######-----######-----
# CORE FUNCTION TO SUM FILE SIZES IN GB
# -----######-----######-----######-----######-----

import os

def _size_2304_i1_GET_total_gb(df):
    """
    Sums the size of all valid files in df['Path'] and returns total in GB (rounded).
    Prints TQM logs.
    """
    total_bytes = 0
    valid_count = 0

    for path in df['Location']:
        if isinstance(path, str) and os.path.isfile(path):
            try:
                total_bytes += os.path.getsize(path)
                valid_count += 1
            except:
                continue

    total_gb = total_bytes / (1024 ** 3)

    print(f"\nTQM ✅ Completed file scan:")
    print(f"   • Valid Files: {valid_count}")
    print(f"   • Total Size: {total_gb:.2f} GB")

    return round(total_gb, 2)

In [19]:
txt_path = "/Users/yerik/Downloads/col_out_arch.txt"
df = _readfix_2304_txt_GET_df_autoclean(txt_path)

print("\n🔎 COLUMN NAMES:")
for i, col in enumerate(df.columns):
    print(f"{i:>2}: '{col}'")
df.head()


🧠 Most common column count: 21
📊 Full column count distribution:
1 columns → 34 lines
4 columns → 1 lines
5 columns → 17 lines
17 columns → 17 lines
18 columns → 1 lines
21 columns → 4826 lines

🔎 COLUMN NAMES:
 0: '#'
 1: 'File Type'
 2: 'Bitrate'
 3: 'BPM'
 4: 'Key'
 5: 'Track Title'
 6: 'Artist'
 7: 'Artwork'
 8: 'Genre'
 9: 'Label'
10: 'Remixer'
11: 'Release Date'
12: 'Year'
13: 'Location'
14: 'Rating'
15: 'File Name'
16: 'Comments'
17: 'Album'
18: 'Bitdepth'
19: 'Time'
20: 'Date Added'


,#,File Type,Bitrate,BPM,Key,Track Title,Artist,Artwork,Genre,Label,...,Release Date,Year,Location,Rating,File Name,Comments,Album,Bitdepth,Time,Date Added
0,1,WAV,1411 kbps,130.00,12A,Ha Dub(2PeKes Vogue Remix),2PeKes,,,,...,,0,/Users/yerik/Music/_0_OLD_SOURCE/_rk_music_35/...,,b9AOl-8_4Bpm_131_0Key_8B_5cnt-rnk_8B3B8B-123oc...,,,16,03:34,2024-08-08
1,2,AIFF,1411 kbps,131.00,6B,Big Moma Tech (Original Mix),Kyle Hall,,Tech House,DistroKid,...,2023-02-17,2023,/Users/yerik/Music/_0_OLD_SOURCE/_rk_music_51/...,,Big_Moma_Tech-BY-Kyle_Hall_(Original_Mix)_KEY_...,,Baci Ballers EP,16,05:39,2023-10-17
2,3,MP3,192 kbps,131.00,6A,Din Daa Daa (Clean),Baltimore Club,,B-More,,...,,0,/Users/yerik/Music/_0_OLD_SOURCE/_rk_music_130...,*,Din Daa Daa (Clean).mp3,,Bmore,16,03:55,2023-08-15
3,4,MP3,320 kbps,84.00,11B,Freak Hoe,Future,,Rap,Epic,...,,2015,/Users/yerik/Music/_0_OLD_SOURCE/_rk_music_133...,,07. Freak Hoe.mp3,,Dirty Sprite 2 (DS2),16,02:54,2023-08-15
4,5,MP3,320 kbps,127.00,1A,La Cocina Del Cabron (Original Mix),"Lee Van Dowski, Glimpse",,Electronica,Global Underground,...,2010-01-31,2010,/Users/yerik/Music/_0_OLD_SOURCE/_rk_music_193...,,"La_Cocina_Del_Cabron-BY-Lee_Van_Dowski,_Glimps...",,Global Underground #38: Carl Cox - Black Rock ...,16,07:11,2024-09-25


# copy files into a new folder 

In [20]:
total_gb = _size_2304_i1_GET_total_gb(df)


TQM ✅ Completed file scan:
   • Valid Files: 4841
   • Total Size: 83.50 GB


In [27]:
# -----######-----###### FIXED COPY LOOP: CORRECT TUPLE UNPACKING -----######-----######
import os
import shutil
import pandas as pd
from tqdm import tqdm

def _fileops_2404_chunkedcopy_GET_genre_sorted_chunks(df, dest_base_path):
    """
    Sorts df by Genre, copies files in chunks of 50 to numbered folders with no filename collisions.
    Adds 'Copied_To' column and saves a CSV log. TQM bar active. No overwrite, all copied.
    """
    tqdm.pandas()
    os.makedirs(dest_base_path, exist_ok=True)
    df = df.copy()

    df = df.sort_values(by='Genre').reset_index(drop=True)
    df['file_name'] = df['Location'].apply(os.path.basename)
    
    copied_to = [None] * len(df)
    folder_idx = 1
    chunk_files = set()
    current_chunk = []

    tq_bar = tqdm(total=len(df), desc="💽 TQM COPY FLOW", ncols=100)

    for idx, row in df.iterrows():
        src = row['Location']
        fname = row['file_name']

        # Start new chunk if needed
        if fname in chunk_files or len(current_chunk) >= 50:
            dest_folder = os.path.join(dest_base_path, f"_24_ARCH_rk_{folder_idx}")
            os.makedirs(dest_folder, exist_ok=True)
            for path, name, i in current_chunk:
                dest_path = os.path.join(dest_folder, name)
                try:
                    shutil.copy2(path, dest_path)
                    copied_to[i] = dest_path
                    tq_bar.update(1)
                except Exception as e:
                    tqdm.write(f"❌ ERROR copying {path} → {dest_path} | {e}")
            tqdm.write(f"📦 Finished chunk: _24_ARCH_rk_{folder_idx} ({len(current_chunk)} files)")

            # Reset chunk
            folder_idx += 1
            chunk_files = set()
            current_chunk = []

        chunk_files.add(fname)
        current_chunk.append((src, fname, idx))

    # Final chunk flush
    if current_chunk:
        dest_folder = os.path.join(dest_base_path, f"_24_ARCH_rk_{folder_idx}")
        os.makedirs(dest_folder, exist_ok=True)
        for path, name, i in current_chunk:
            dest_path = os.path.join(dest_folder, name)
            try:
                shutil.copy2(path, dest_path)
                copied_to[i] = dest_path
                tq_bar.update(1)
            except Exception as e:
                tqdm.write(f"❌ ERROR copying {path} → {dest_path} | {e}")
        tqdm.write(f"📦 Final chunk: _24_ARCH_rk_{folder_idx} ({len(current_chunk)} files)")

    tq_bar.close()
    df['Copied_To'] = copied_to

    # Save log
    log_path = os.path.join(dest_base_path, "copied_log.csv")
    df.to_csv(log_path, index=False)
    print(f"\n✅ Done. Log saved at: {log_path}")
    return df


In [28]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
updated_df = _fileops_2404_chunkedcopy_GET_genre_sorted_chunks(
    df,
    "/Volumes/MY1TB/_24_ARCH_SONGS"
)





💽 TQM COPY FLOW:   0%|                                                    | 0/4895 [00:00<?, ?it/s]


💽 TQM COPY FLOW:   0%|                                          | 1/4895 [00:02<3:33:27,  2.62s/it]


💽 TQM COPY FLOW:   0%|                                          | 2/4895 [00:03<1:55:21,  1.41s/it]


💽 TQM COPY FLOW:   0%|                                          | 3/4895 [00:03<1:25:49,  1.05s/it]


💽 TQM COPY FLOW:   0%|                                          | 4/4895 [00:04<1:02:55,  1.30it/s]


💽 TQM COPY FLOW:   0%|                                            | 5/4895 [00:04<54:05,  1.51it/s]


💽 TQM COPY FLOW:   0%|                                            | 6/4895 [00:05<50:27,  1.61it/s]


💽 TQM COPY FLOW:   0%|                                            | 7/4895 [00:06<59:04,  1.38it/s]


💽 TQM COPY FLOW:   0%|                                          | 8/4895 [00:06<1:00:32,  1.35it/s]


💽 TQM COPY FLOW:   0%|                                            | 9/4895 [00:

📦 Finished chunk: _24_ARCH_rk_1 (50 files)





💽 TQM COPY FLOW:   1%|▍                                          | 51/4895 [00:31<51:13,  1.58it/s]


💽 TQM COPY FLOW:   1%|▍                                          | 52/4895 [00:31<51:31,  1.57it/s]


💽 TQM COPY FLOW:   1%|▍                                          | 53/4895 [00:32<42:32,  1.90it/s]


💽 TQM COPY FLOW:   1%|▍                                          | 54/4895 [00:32<39:34,  2.04it/s]


💽 TQM COPY FLOW:   1%|▍                                          | 55/4895 [00:32<33:52,  2.38it/s]


💽 TQM COPY FLOW:   1%|▍                                          | 56/4895 [00:33<42:15,  1.91it/s]


💽 TQM COPY FLOW:   1%|▌                                          | 57/4895 [00:34<43:57,  1.83it/s]


💽 TQM COPY FLOW:   1%|▌                                          | 58/4895 [00:35<55:29,  1.45it/s]


💽 TQM COPY FLOW:   1%|▌                                          | 59/4895 [00:35<43:16,  1.86it/s]


💽 TQM COPY FLOW:   1%|▌                                          | 60/4895 [00:

❌ ERROR copying  → /Volumes/MY1TB/_24_ARCH_SONGS/_24_ARCH_rk_2/ | [Errno 2] No such file or directory: ''





💽 TQM COPY FLOW:   2%|▊                                          | 95/4895 [00:49<21:40,  3.69it/s]


💽 TQM COPY FLOW:   2%|▊                                          | 96/4895 [00:50<34:40,  2.31it/s]


💽 TQM COPY FLOW:   2%|▊                                          | 97/4895 [00:51<33:45,  2.37it/s]


💽 TQM COPY FLOW:   2%|▊                                          | 98/4895 [00:51<30:18,  2.64it/s]


                                                                                                    

                                                                                              


                                                                                           
💽 TQM COPY FLOW:   2%|▊                                          | 93/4895 [06:49<31:39,  2.53it/s]

💽 TQM COPY FLOW:   0%|                                                    | 0/4895 [01:19<?, ?it/s]


💽 TQM COPY FLOW:   0%|                                                    | 0/4895 [01:25<?, ?i

...

📦 Finished chunk: _24_ARCH_rk_160 (11 files)





💽 TQM COPY FLOW:  98%|██████████████████████████████████████▍| 4820/4895 [1:47:00<01:21,  1.08s/it]


💽 TQM COPY FLOW:  98%|██████████████████████████████████████▍| 4821/4895 [1:47:01<01:13,  1.00it/s]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▍| 4822/4895 [1:47:02<01:16,  1.05s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▍| 4823/4895 [1:47:03<01:24,  1.17s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▍| 4824/4895 [1:47:05<01:26,  1.22s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▍| 4825/4895 [1:47:07<01:40,  1.44s/it]


                                                                                                    

                                                                                              


                                                                                           
💽 TQM COPY FLOW:   2%|▊                                        | 93/4895 [1:53:06<31:39,  2.53

📦 Finished chunk: _24_ARCH_rk_161 (7 files)





💽 TQM COPY FLOW:  99%|██████████████████████████████████████▍| 4827/4895 [1:47:09<01:33,  1.38s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▍| 4828/4895 [1:47:11<01:31,  1.37s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▍| 4829/4895 [1:47:12<01:31,  1.38s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▍| 4830/4895 [1:47:14<01:32,  1.42s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▍| 4831/4895 [1:47:14<01:20,  1.26s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▍| 4832/4895 [1:47:15<01:11,  1.14s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▌| 4833/4895 [1:47:16<01:08,  1.10s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▌| 4834/4895 [1:47:17<01:07,  1.11s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▌| 4835/4895 [1:47:19<01:20,  1.34s/it]


💽 TQM COPY FLOW:  99%|██████████████████████████████████████▌| 4836/4895 [1:47:

📦 Final chunk: _24_ARCH_rk_162 (15 files)

✅ Done. Log saved at: /Volumes/MY1TB/_24_ARCH_SONGS/copied_log.csv


In [29]:
50*162

8100